# 3-Fold CV: Joint LoRA + Soft-Prompt Fine-Tuning

Combines LoRA fine-tuning of BioCLIP 2's image encoder with soft-prompt tuning of the text encoder. Both are trained jointly via CLIP-style contrastive loss (cross-entropy on cosine similarities between LoRA-modified image features and soft-prompt-tuned text features).

**Classification mechanism:** cosine similarity argmax (the original CLIP architecture, not logistic regression). Predictions are restricted to the 9 classes with n>=50 because soft-prompts are only trained for those.

**Hyperparameters:**
- LoRA: copied from sweep winner (updated from `01_sweep.ipynb` results).
- Soft-prompt: N=4 slots per class, matching the soft-prompt experiment winner.

**Outputs:**
- `output_cv_joint/per_fold_per_class_results.csv` — per-fold per-class metrics.
- `output_cv_joint/per_class_summary_mean_std.csv` — per-class mean ± std.
- `output_cv_joint/headline_metrics_mean_std.csv` — aggregate metrics with mean ± std.

**Note on classification range:** since the joint model can only predict the 9 trained classes, the 5 excluded classes (Mecoptera, Blattodea, Ixodida, Orthoptera, Raphidioptera) will always be misclassified. This is the inherent cost of closed-set joint training and is reported transparently.

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader
import open_clip
from peft import LoraConfig, get_peft_model

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    "model_name": "hf-hub:imageomics/bioclip-2",
    "embedding_dim": 768,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_cv_joint",

    "image_id_col": "image_id",
    "label_col": "label",

    "k_folds": 3,
    "random_state": 0,
    "min_class_size": 50,

    # ============================================
    # WINNING HYPERPARAMETERS — updated according to 01_sweep.ipynb output
    # ============================================
    "lr": 5e-4,
    "r": 16,
    "epochs": 2,
    # ============================================

    "lora_target_modules": ["out_proj", "c_fc", "c_proj"],
    "lora_alpha_factor": 2,
    "lora_dropout": 0.05,

    # Soft-prompt slots per class (from earlier soft-prompt experiment winner)
    "n_slots": 4,

    "batch_size": 64,
    "weight_decay": 1e-4,
    "num_workers": 4,

    # Per-class taxonomy templates (matches soft-prompt experiment)
    "class_label_templates": {
        "Blattodea":     "Animalia Arthropoda Insecta Blattodea",
        "Coleoptera":    "Animalia Arthropoda Insecta Coleoptera",
        "Diptera":       "Animalia Arthropoda Insecta Diptera",
        "Hemiptera":     "Animalia Arthropoda Insecta Hemiptera",
        "Hymenoptera":   "Animalia Arthropoda Insecta Hymenoptera",
        "Lepidoptera":   "Animalia Arthropoda Insecta Lepidoptera",
        "Mecoptera":     "Animalia Arthropoda Insecta Mecoptera",
        "Neuroptera":    "Animalia Arthropoda Insecta Neuroptera",
        "Orthoptera":    "Animalia Arthropoda Insecta Orthoptera",
        "Plecoptera":    "Animalia Arthropoda Insecta Plecoptera",
        "Raphidioptera": "Animalia Arthropoda Insecta Raphidioptera",
        "Trichoptera":   "Animalia Arthropoda Insecta Trichoptera",
        "Araneae":       "Animalia Arthropoda Arachnida Araneae",
        "Ixodida":       "Animalia Arthropoda Arachnida Ixodida",
    },
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"LoRA hyperparameters: lr={CONFIG['lr']}, r={CONFIG['r']}, epochs={CONFIG['epochs']}")
print(f"Soft-prompt slots: N={CONFIG['n_slots']}")
print(f"Output: {CONFIG['output_dir']}")

LoRA hyperparameters: lr=0.0005, r=16, epochs=2
Soft-prompt slots: N=4
Output: /workspace/thesis/output_cv_joint


## Helper functions: dataset and path indexing

In [3]:
def build_path_index(image_root):
    """Scanning local image directory, building {image_id: full_path}."""
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


class ImageOrderDataset(Dataset):
    """Dataset of (image, label) pairs loaded from disk."""
    def __init__(self, image_paths, labels, preprocess):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.preprocess(img)
        return img, self.labels[idx]

## Text encoder with custom embeddings

Replicates `open_clip`'s text forward path manually, accepting pre-built token embeddings (rather than token IDs). This is needed because soft-prompts insert learnable vectors at specific positions in the embedding sequence.

Uses `batch_first=True` convention (no permutation), iterating through resblocks manually with the model's own attention mask. This was empirically validated to match `model.encode_text` exactly (max diff = 0.000000 in earlier diagnostic).

In [4]:
class TextEncoderWithCustomEmbeddings(nn.Module):
    """Wraps open_clip's text transformer to accept custom token embeddings."""
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.attn_mask = clip_model.attn_mask

    def forward(self, prompts, tokenized_prompts):
        """
        prompts: (n_cls, seq_len, embedding_dim) — token embeddings with slots inserted
        tokenized_prompts: (n_cls, seq_len) — original token IDs, used for EOS lookup
        """
        cast_dtype = self.transformer.get_cast_dtype()
        x = prompts + self.positional_embedding.to(cast_dtype)
        # No permute — open_clip uses batch_first=True
        for resblock in self.transformer.resblocks:
            x = resblock(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)
        # Take features from EOS token position
        x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)]
        # Apply text projection (handle both Linear and tensor variants)
        if self.text_projection is not None:
            if isinstance(self.text_projection, nn.Linear):
                x = self.text_projection(x)
            else:
                x = x @ self.text_projection
        return x

## Soft-prompt learner

Stores `n_slots` learnable vectors per class. The class label string (taxonomic hierarchy) is tokenized with placeholder tokens at the slot positions; at training time, those placeholders are replaced with the trainable slot vectors.

Structure of each per-class sequence:
```
[SOS] [slot_1] [slot_2] ... [slot_N] [class_label_tokens] [EOS] [padding]
```

Only `ctx` (the slot vectors) is trainable. SOS/class label/EOS/padding are frozen token embeddings.

In [5]:
class PromptLearner(nn.Module):
    """Learns soft-prompt slot vectors, one set per class."""
    def __init__(self, classnames, clip_model, tokenizer, n_slots, label_templates, dtype, device):
        super().__init__()
        self.n_cls = len(classnames)
        self.n_slots = n_slots
        self.classnames = classnames

        try:
            base_strings = [label_templates[name] for name in classnames]
        except KeyError as e:
            raise KeyError(f"No template for class {e}")

        # Initialize slot vectors with small random values
        ctx_vectors = torch.empty(self.n_cls, n_slots, CONFIG["embedding_dim"], dtype=dtype)
        nn.init.normal_(ctx_vectors, std=0.02)
        self.ctx = nn.Parameter(ctx_vectors)

        # Tokenize with placeholder slots ("X X X X <class_label>")
        placeholder = " ".join(["X"] * n_slots)
        prompts_with_placeholder = [f"{placeholder} {s}" for s in base_strings]
        tokenized_prompts = tokenizer(prompts_with_placeholder).to(device)

        # Get token embeddings (these are frozen; only slots will be trainable)
        with torch.no_grad():
            embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)

        # SOS prefix and suffix (class label + EOS + padding) are frozen buffers
        self.register_buffer("token_prefix", embedding[:, :1, :])  # SOS token
        self.register_buffer("token_suffix", embedding[:, 1 + n_slots:, :])  # rest
        self.register_buffer("tokenized_prompts", tokenized_prompts)

    def forward(self):
        """Construct the full per-class embeddings with current slot vectors."""
        prompts = torch.cat([self.token_prefix, self.ctx, self.token_suffix], dim=1)
        return prompts, self.tokenized_prompts

## Joint training and evaluation functions

`train_joint`: trains LoRA + soft-prompts simultaneously with CLIP-style contrastive loss.

`evaluate_joint`: predicts via cosine similarity between LoRA-modified image features and soft-prompt-tuned text features (argmax over 9 trained classes).

In [6]:
def train_joint(model, train_loader, classnames_eligible, tokenizer, config, fold_idx):
    """Jointly train LoRA on image encoder and soft-prompts on text side.

    Loss: cross-entropy on cosine similarity between modified image features and
    soft-prompted text features, scaled by CLIP's logit_scale.
    """
    device = config["device"]
    dtype = model.transformer.get_cast_dtype()

    # Applying LoRA to image encoder
    lora_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["r"] * config["lora_alpha_factor"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["lora_target_modules"],
        bias="none",
    )
    visual = get_peft_model(model.visual, lora_config).to(device)
    visual.print_trainable_parameters()

    # Building soft-prompt learner (only for eligible classes)
    prompt_learner = PromptLearner(
        classnames=classnames_eligible,
        clip_model=model,
        tokenizer=tokenizer,
        n_slots=config["n_slots"],
        label_templates=config["class_label_templates"],
        dtype=dtype,
        device=device,
    ).to(device)

    # Text encoder wrapper
    text_encoder = TextEncoderWithCustomEmbeddings(model).to(device)

    # Freezing text encoder weights (only soft-prompt slots train on text side)
    for param in text_encoder.parameters():
        param.requires_grad = False
    for param in model.token_embedding.parameters():
        param.requires_grad = False

    # Using CLIP's pretrained logit_scale (frozen for stability)
    logit_scale = model.logit_scale.detach().exp().to(device)

    # Trainable params: LoRA + soft-prompt slots
    trainable = [p for p in visual.parameters() if p.requires_grad]
    trainable += [prompt_learner.ctx]
    n_trainable = sum(p.numel() for p in trainable)
    print(f"    Total trainable params: {n_trainable:,}")

    optimizer = torch.optim.AdamW(trainable, lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    print(f"    Joint training: LoRA (lr={config['lr']}, r={config['r']}) + "
          f"soft-prompts (N={config['n_slots']} slots, {len(classnames_eligible)} classes)")

    for epoch in range(config["epochs"]):
        visual.train()
        prompt_learner.train()
        total_loss, n_batches = 0.0, 0
        ep_start = time.time()

        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Forwarding through LoRA-modified image encoder
            image_features = visual(imgs)
            image_features = F.normalize(image_features, dim=-1)

            # Forwarding through soft-prompted text encoder
            prompts, tokenized_prompts = prompt_learner()
            text_features = text_encoder(prompts, tokenized_prompts)
            text_features = F.normalize(text_features, dim=-1)

            # CLIP-style logits: scaled cosine similarity
            logits = logit_scale * image_features @ text_features.t()
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

            if batch_idx % 100 == 0:
                print(f"      Fold {fold_idx+1} Ep {epoch+1} batch {batch_idx}/{len(train_loader)}: "
                      f"loss={loss.item():.4f}")

        scheduler.step()
        print(f"      Fold {fold_idx+1} Ep {epoch+1}: avg loss={total_loss/n_batches:.4f} "
              f"({(time.time()-ep_start)/60:.1f} min)")

    return visual, prompt_learner, text_encoder, logit_scale


def evaluate_joint(visual, prompt_learner, text_encoder, logit_scale, val_dataset, config):
    """Predict classes via cosine similarity to soft-prompt-tuned text features."""
    device = config["device"]
    visual.eval()
    prompt_learner.eval()

    loader = DataLoader(
        val_dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )

    # Computing text features once (constant across val batches)
    with torch.no_grad():
        prompts, tokenized_prompts = prompt_learner()
        text_features = text_encoder(prompts, tokenized_prompts)
        text_features = F.normalize(text_features, dim=-1)

    all_preds = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device, non_blocking=True)
            image_features = visual(imgs)
            image_features = F.normalize(image_features, dim=-1)
            logits = logit_scale * image_features @ text_features.t()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)

    return np.concatenate(all_preds, axis=0)

## Load model, tokenizer, and metadata

In [7]:
print("Loading BioCLIP 2...")
model, _, preprocess = open_clip.create_model_and_transforms(CONFIG["model_name"])
tokenizer = open_clip.get_tokenizer(CONFIG["model_name"])
model = model.to(CONFIG["device"])
print("Done.")

Loading BioCLIP 2...


Done.


In [8]:
# Building path index from local image directory
path_index = build_path_index(CONFIG["image_root"])

# Loading and merging metadata
meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)
print(f"Matched: {len(matched)} crops")

# Setting up label indexing
y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_indices = [class_to_idx[c] for c in eligible_classes]
eligible_class_set = set(eligible_classes)

# Soft-prompts only exist for eligible classes — building separate indexing
eligible_classnames = sorted(eligible_classes)
eligible_name_to_eligible_idx = {c: i for i, c in enumerate(eligible_classnames)}
# Maps original class index -> position in eligible_classnames, or -1 if not eligible
orig_to_eligible_idx = np.array([
    eligible_name_to_eligible_idx.get(c, -1) for c in classnames
])

print(f"\nClasses ({len(classnames)}): {classnames}")
print(f"Eligible for joint training (n>=50): {eligible_classnames}")
print(f"Excluded (cannot be predicted by joint model): "
      f"{[c for c in classnames if c not in eligible_class_set]}")

Mapped 48095 image_ids
Matched: 39787 crops

Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Eligible for joint training (n>=50): ['Araneae', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Lepidoptera', 'Neuroptera', 'Plecoptera', 'Trichoptera']
Excluded (cannot be predicted by joint model): ['Blattodea', 'Ixodida', 'Mecoptera', 'Orthoptera', 'Raphidioptera']


## Set up 3-fold stratified cross-validation

Classes with fewer than `k_folds` examples are kept in the training set for every fold (Ixodida, Raphidioptera at minimum). They cannot appear in any val fold.

In [9]:
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= CONFIG["k_folds"]
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

print(f"Splittable samples (n>={CONFIG['k_folds']}): {len(splittable_indices)}")
print(f"Unsplittable (n<{CONFIG['k_folds']}, kept in train): {len(unsplittable_indices)}")

skf = StratifiedKFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=CONFIG["random_state"])
raw_splits = list(skf.split(splittable_indices, y[splittable_indices]))

fold_splits = []
for train_rel, val_rel in raw_splits:
    train_idx = np.concatenate([splittable_indices[train_rel], unsplittable_indices])
    val_idx = splittable_indices[val_rel]
    fold_splits.append((train_idx, val_idx))

for i, (tr, va) in enumerate(fold_splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(va)}")

Splittable samples (n>=3): 39784
Unsplittable (n<3, kept in train): 3
  Fold 1: train=26525, val=13262
  Fold 2: train=26526, val=13261
  Fold 3: train=26526, val=13261


## Running cross-validation

For each fold:
1. Filtering training set to eligible classes only (joint model can only learn these).
2. Remapping training labels from original 14-class indexing to 9-class eligible indexing.
3. Training joint LoRA + soft-prompts on the eligible training set.
4. Evaluating on the full validation set (all 14 classes). Predictions are remapped back to original indexing for reporting.

Excluded classes will appear in the per-class report with zero recall (the model cannot predict them).

In [10]:
for name, module in model.visual.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name)

transformer.resblocks.0.attn.out_proj
transformer.resblocks.0.mlp.c_fc
transformer.resblocks.0.mlp.c_proj
transformer.resblocks.1.attn.out_proj
transformer.resblocks.1.mlp.c_fc
transformer.resblocks.1.mlp.c_proj
transformer.resblocks.2.attn.out_proj
transformer.resblocks.2.mlp.c_fc
transformer.resblocks.2.mlp.c_proj
transformer.resblocks.3.attn.out_proj
transformer.resblocks.3.mlp.c_fc
transformer.resblocks.3.mlp.c_proj
transformer.resblocks.4.attn.out_proj
transformer.resblocks.4.mlp.c_fc
transformer.resblocks.4.mlp.c_proj
transformer.resblocks.5.attn.out_proj
transformer.resblocks.5.mlp.c_fc
transformer.resblocks.5.mlp.c_proj
transformer.resblocks.6.attn.out_proj
transformer.resblocks.6.mlp.c_fc
transformer.resblocks.6.mlp.c_proj
transformer.resblocks.7.attn.out_proj
transformer.resblocks.7.mlp.c_fc
transformer.resblocks.7.mlp.c_proj
transformer.resblocks.8.attn.out_proj
transformer.resblocks.8.mlp.c_fc
transformer.resblocks.8.mlp.c_proj
transformer.resblocks.9.attn.out_proj
transfor

In [11]:
all_results = []
overall_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{CONFIG['k_folds']}\n{'='*70}")
    fold_start = time.time()

    # Filtering training set to eligible classes
    train_mask = np.isin(y[train_idx], eligible_class_indices)
    train_idx_eligible = train_idx[train_mask]

    # Remaping training labels to eligible indexing (0..n_eligible-1)
    train_paths_eligible = image_paths[train_idx_eligible]
    train_labels_eligible_orig = y[train_idx_eligible]
    train_labels_eligible_remapped = orig_to_eligible_idx[train_labels_eligible_orig]
    assert (train_labels_eligible_remapped >= 0).all(), "Training set contains non-eligible classes"

    val_paths = image_paths[val_idx]
    val_labels_orig = y[val_idx]  # keep in original 14-class indexing

    print(f"  Train (eligible classes): {len(train_idx_eligible)}")
    print(f"  Val (all classes for eval): {len(val_idx)}")

    train_dataset = ImageOrderDataset(train_paths_eligible, train_labels_eligible_remapped, preprocess)
    val_dataset = ImageOrderDataset(val_paths, val_labels_orig, preprocess)

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG["batch_size"],
        shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True,
    )

    # Reloading fresh BioCLIP 2 (avoid PEFT adapter stacking)
    print("\n  Reloading fresh BioCLIP 2...")
    reload_start = time.time()
    model_fresh, _, _ = open_clip.create_model_and_transforms(CONFIG["model_name"])
    model_fresh = model_fresh.to(CONFIG["device"])
    print(f"  Reload took {time.time() - reload_start:.1f}s")

    # Joint training
    train_start = time.time()
    visual, prompt_learner, text_encoder, logit_scale = train_joint(
        model_fresh, train_loader, eligible_classnames, tokenizer, CONFIG, fold_idx
    )
    print(f"\n  Joint training time: {timedelta(seconds=int(time.time() - train_start))}")

    # Evaluation
    print("  Evaluating on val set...")
    eval_start = time.time()
    # Predictions are in eligible indexing (0..n_eligible-1)
    preds_eligible_idx = evaluate_joint(visual, prompt_learner, text_encoder, logit_scale, val_dataset, CONFIG)
    # Map back to original 14-class indexing for reporting
    preds_orig_idx = np.array([
        class_to_idx[eligible_classnames[p]] for p in preds_eligible_idx
    ])
    print(f"  Eval time: {timedelta(seconds=int(time.time() - eval_start))}")

    # Classification report (across all 14 classes; excluded classes get zero recall)
    report = classification_report(
        val_labels_orig, preds_orig_idx,
        labels=list(range(len(classnames))),
        target_names=classnames,
        output_dict=True, zero_division=0,
    )

    for cls_name in classnames:
        stats = report.get(cls_name, {})
        all_results.append({
            "method": "joint_lora_softprompt",
            "fold": fold_idx,
            "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_joint": cls_name in eligible_class_set,
        })

    # Saving intermediate results
    pd.DataFrame(all_results).to_csv(
        Path(CONFIG["output_dir"]) / "per_fold_per_class_results.csv", index=False
    )

    # Cleanup
    del visual, prompt_learner, text_encoder, model_fresh
    torch.cuda.empty_cache()

    print(f"\n  Fold {fold_idx+1} total: {timedelta(seconds=int(time.time() - fold_start))}")

print(f"\n{'='*70}")
print(f"CV COMPLETE — Total: {timedelta(seconds=int(time.time() - overall_start))}")
print(f"{'='*70}")


FOLD 1/3
  Train (eligible classes): 26480
  Val (all classes for eval): 13262

  Reloading fresh BioCLIP 2...
  Reload took 20.0s
trainable params: 4,718,592 || all params: 308,684,800 || trainable%: 1.5286
    Total trainable params: 4,746,240
    Joint training: LoRA (lr=0.0005, r=16) + soft-prompts (N=4 slots, 9 classes)
      Fold 1 Ep 1 batch 0/414: loss=5.5839
      Fold 1 Ep 1 batch 100/414: loss=0.1174
      Fold 1 Ep 1 batch 200/414: loss=0.0832
      Fold 1 Ep 1 batch 300/414: loss=0.0913
      Fold 1 Ep 1 batch 400/414: loss=0.0658
      Fold 1 Ep 1: avg loss=0.1876 (11.7 min)
      Fold 1 Ep 2 batch 0/414: loss=0.1082
      Fold 1 Ep 2 batch 100/414: loss=0.0143
      Fold 1 Ep 2 batch 200/414: loss=0.0182
      Fold 1 Ep 2 batch 300/414: loss=0.0901
      Fold 1 Ep 2 batch 400/414: loss=0.0553
      Fold 1 Ep 2: avg loss=0.0562 (11.8 min)

  Joint training time: 0:23:29
  Evaluating on val set...
  Eval time: 0:02:13

  Fold 1 total: 0:26:02

FOLD 2/3
  Train (eligible c

## Aggregate results: per-class mean and std across folds

In [12]:
results_df = pd.DataFrame(all_results)

summary_rows = []
for cls_name in classnames:
    class_data = results_df[results_df["class"] == cls_name]
    summary_rows.append({
        "class": cls_name,
        "support_total": int(class_data["support"].sum()),
        "f1_mean": round(class_data["f1"].mean(), 4),
        "f1_std": round(class_data["f1"].std(ddof=1), 4),
        "precision_mean": round(class_data["precision"].mean(), 4),
        "precision_std": round(class_data["precision"].std(ddof=1), 4),
        "recall_mean": round(class_data["recall"].mean(), 4),
        "recall_std": round(class_data["recall"].std(ddof=1), 4),
        "trained_in_joint": cls_name in eligible_class_set,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_summary_mean_std.csv", index=False)
summary_df

,class,support_total,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,trained_in_joint
0,Araneae,564,0.9817,0.0185,0.9982,0.0031,0.9663,0.0378,True
1,Blattodea,23,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
2,Coleoptera,1143,0.9061,0.0076,0.9155,0.0309,0.8976,0.0160,True
3,Diptera,31913,0.9892,0.0010,0.9893,0.0013,0.9890,0.0033,True
4,Hemiptera,775,0.8833,0.0249,0.8628,0.0586,0.9071,0.0155,True
5,Hymenoptera,3326,0.9393,0.0068,0.9283,0.0219,0.9510,0.0090,True
6,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,Lepidoptera,1744,0.9788,0.0163,0.9936,0.0043,0.9650,0.0345,True
8,Mecoptera,34,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
9,Neuroptera,115,0.7280,0.0591,0.7321,0.0381,0.7402,0.1604,True


## Headline aggregate metrics (mean ± std across folds)

In [13]:
per_fold_weighted = []
per_fold_macro_all = []
per_fold_macro_eligible = []
per_fold_hemi = []
per_fold_cole = []

for fold in range(CONFIG["k_folds"]):
    fold_data = results_df[results_df["fold"] == fold]
    per_fold_weighted.append(np.average(fold_data["f1"], weights=fold_data["support"]))
    per_fold_macro_all.append(fold_data["f1"].mean())
    eligible_data = fold_data[fold_data["class"].isin(eligible_class_set)]
    per_fold_macro_eligible.append(eligible_data["f1"].mean())
    per_fold_hemi.append(fold_data[fold_data["class"] == "Hemiptera"]["f1"].iloc[0])
    per_fold_cole.append(fold_data[fold_data["class"] == "Coleoptera"]["f1"].iloc[0])

headline_df = pd.DataFrame([{
    "method": "joint_lora_softprompt",
    "weighted_f1_mean": round(np.mean(per_fold_weighted), 4),
    "weighted_f1_std": round(np.std(per_fold_weighted, ddof=1), 4),
    "macro_f1_all_mean": round(np.mean(per_fold_macro_all), 4),
    "macro_f1_all_std": round(np.std(per_fold_macro_all, ddof=1), 4),
    "macro_f1_eligible_mean": round(np.mean(per_fold_macro_eligible), 4),
    "macro_f1_eligible_std": round(np.std(per_fold_macro_eligible, ddof=1), 4),
    "Hemiptera_f1_mean": round(np.mean(per_fold_hemi), 4),
    "Hemiptera_f1_std": round(np.std(per_fold_hemi, ddof=1), 4),
    "Coleoptera_f1_mean": round(np.mean(per_fold_cole), 4),
    "Coleoptera_f1_std": round(np.std(per_fold_cole, ddof=1), 4),
}])
headline_df.to_csv(Path(CONFIG["output_dir"]) / "headline_metrics_mean_std.csv", index=False)
headline_df

,method,weighted_f1_mean,weighted_f1_std,macro_f1_all_mean,macro_f1_all_std,macro_f1_eligible_mean,macro_f1_eligible_std,Hemiptera_f1_mean,Hemiptera_f1_std,Coleoptera_f1_mean,Coleoptera_f1_std
0,joint_lora_softprompt,0.9771,0.0022,0.5782,0.005,0.8995,0.0077,0.8833,0.0249,0.9061,0.0076


## Summary

Joint LoRA + soft-prompt training results saved to:
- `per_fold_per_class_results.csv` — granular per-fold per-class
- `per_class_summary_mean_std.csv` — per-class mean ± std
- `headline_metrics_mean_std.csv` — aggregate metrics

Goal: Comparing against `02_cv_lora.ipynb` results (LoRA + linear probe) to determine whether keeping the CLIP-style architecture (cosine similarity classification) provides any benefit when both encoders are jointly adapted.

## Output files summary

The notebook generates:

| File | Purpose |
|---|---|
| `per_fold_per_class_results.csv` | Granular per-fold per-class precision/recall/F1 |
| `per_class_summary_mean_std.csv` | Per-class aggregates with mean ± std across folds |
| `headline_metrics_mean_std.csv` | Top-line aggregates (weighted F1, macro F1, etc.) with mean ± std |

**Note:** unlike `02_cv_lora.ipynb`, this notebook does not save features. Joint LoRA + soft-prompt models adapt both image and text encoders simultaneously, so the cached image features alone wouldn't be sufficient to characterize the learned representation — one needs to also save the soft-prompt vectors and text features. Since the joint experiment is primarily a comparison point against LoRA + linear probe (rather than a basis for downstream analysis), feature saving was omitted.

The 5 excluded classes (Mecoptera, Blattodea, Ixodida, Orthoptera, Raphidioptera) receive F1 = 0 by construction in this experiment, since the joint model can only predict classes for which soft-prompts were trained. See methodological note in the manuscript for full discussion.
